In [2]:
import optuna
import torch
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torchmetrics import F1Score
import warnings
from GradientGang.Pipeline.Architectures.LightningAutoencoder import LightningAutoencoder

import dotenv
import os

warnings.filterwarnings('ignore')

from GradientGang.Pipeline.DataLoader.DataLoader import DataModule
from GradientGang.Pipeline.Architectures.Direct import Direct

# Load database configuration
dotenv.load_dotenv(dotenv_path="./../src/GradientGang/Pipeline/Optimizer/.env")
storage = os.getenv("DATABASE_URL")

print("✓ Database configuration loaded")
print(f"Storage: {storage[:21]}..." if storage else "⚠️ No database URL found")

✓ Database configuration loaded
Storage: postgresql://postgres...


# Mega Optuna Notebook - Multi-Architecture Optimization

This notebook performs comprehensive hyperparameter optimization for the Pirate Pain Classification task.

## Features
- **Database Integration**: Results are stored in a PostgreSQL database for persistence and multi-process optimization
- **Multi-Architecture Search**: Supports Direct and Autoencoder architectures with RNN and Conv1d encoders
- **Smart Pruning**: Uses MedianPruner to stop unpromising trials early
- **Resume Capability**: Can continue optimization from where it left off
- **He Initialization**: Proper weight initialization for faster convergence and better performance

## Supported Architectures

### Macro Architectures:
1. **Direct**: End-to-end encoder + classifier
2. **Autoencoder**: Encoder-Decoder with joint reconstruction + classification loss (semi-supervised with test data)

### Encoder Types:
1. **Recurrent (RNN)**: LSTM/GRU with configurable layers, bidirectionality, and dropout
2. **Conv1d**: 1D Convolutional networks with adaptive pooling

### Search Space:
- **Encoder**: Architecture type, hidden dimensions, number of layers, dropout, activation
- **Classifier Head**: Number of layers, hidden dimensions, dropout, activation  
- **Training**: Learning rate, regularization weight, patience, max epochs
- **Autoencoder** (if selected): Reconstruction loss weight

The optimization uses TPE sampler with median pruning for efficient hyperparameter search.

In [3]:
# Fixed data loading parameters
data_params = {
    'data_dir': "../dataset/PirateProcessed/",
    'train_file_name': "pirate_pain_train.csv",
    'train_file_name_labels': "pirate_pain_train_labels.csv",
    'test_file_name': "pirate_pain_test.csv",
    'batch_size': 32,
    'num_workers': 0,
    'val_split': 0.2,
    'shuffle': True,
}

# Initialize data module
dataLoader = DataModule(params=data_params)
dataLoader.setup(stage='fit', includeTestInTrain=True)

trainLoader = dataLoader.train_dataloader()
valLoader = dataLoader.val_dataloader()

print("Data loaders initialized successfully!")
print(f"Training batches: {len(trainLoader)}")
print(f"Validation batches: {len(valLoader)}")

Data loaders initialized successfully!
Training batches: 58
Validation batches: 5


In [4]:
# Display current search space
print("=" * 60)
print("HYPERPARAMETER SEARCH SPACE")
print("=" * 60)
print("\n📊 Macro Architecture:")
print("  • Direct (end-to-end)")
print("  • Autoencoder (semi-supervised with test data)")

print("\n🏗️ Encoder Types:")
print("  • Recurrent: LSTM/GRU")
print("    - Hidden dim: 16-256")
print("    - Layers: 1-3")
print("    - Bidirectional: True/False")
print("    - Dropout: 0.0-0.5")
print("  • Conv1d:")
print("    - Layers: 1-3")
print("    - Kernel size: 3/5/7")
print("    - Channels: 16-128 per layer")
print("    - Stride: 1/2")

print("\n🧠 Classifier Head:")
print("  • Layers: 1-3")
print("  • Hidden dim: 32-256")
print("  • Dropout: 0.0-0.5")
print("  • Activation: ReLU/LeakyReLU/GELU")

print("\n⚙️ Training:")
print("  • Learning rate: 1e-5 to 1e-2 (log scale)")
print("  • Regularization: 1e-6 to 1e-2 (log scale)")
print("  • Patience: 3-15")
print("  • Max epochs: 20-100")

print("\n🔄 Autoencoder (if selected):")
print("  • Reconstruction loss weight: 0.1-0.9")

print("\n" + "=" * 60)

HYPERPARAMETER SEARCH SPACE

📊 Macro Architecture:
  • Direct (end-to-end)
  • Autoencoder (semi-supervised with test data)

🏗️ Encoder Types:
  • Recurrent: LSTM/GRU
    - Hidden dim: 16-256
    - Layers: 1-3
    - Bidirectional: True/False
    - Dropout: 0.0-0.5
  • Conv1d:
    - Layers: 1-3
    - Kernel size: 3/5/7
    - Channels: 16-128 per layer
    - Stride: 1/2

🧠 Classifier Head:
  • Layers: 1-3
  • Hidden dim: 32-256
  • Dropout: 0.0-0.5
  • Activation: ReLU/LeakyReLU/GELU

⚙️ Training:
  • Learning rate: 1e-5 to 1e-2 (log scale)
  • Regularization: 1e-6 to 1e-2 (log scale)
  • Patience: 3-15
  • Max epochs: 20-100

🔄 Autoencoder (if selected):
  • Reconstruction loss weight: 0.1-0.9



In [5]:
def setUpEncoder(trial:optuna.Trial, architectureParameters:dict, datasetInfo:dict):
    # Setup global features encoder
    # Global features is just one binary (isPirate), so we don't need a hidden dimension
    globalEmbeddingDim = 1  # Fixed to 1 since it's just one binary feature
    globalEncoderParams = {
        "activation_function": "LeakyReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": datasetInfo["globalFeaturesShape"][0],
                    "out_features": globalEmbeddingDim,
                    "bias": True,
                }
            },
        ]
    }
    architectureParameters["GlobalFFEncoderParams"] = globalEncoderParams
    
    # Setup time series encoder - now supports RNN and Conv1d
    architectureType = trial.suggest_categorical("architectureType", ["Recurrent", "Conv1d"])
    
    timeSeriesEncoderParams = {}
    
    if architectureType == "Recurrent":
        rnnType = trial.suggest_categorical("rnnType", ["LSTM", "GRU"])
        hiddenDim = trial.suggest_int("hiddenDim", 16, 256)
        numLayers = trial.suggest_int("numLayers", 1, 3)
        bidirectional = trial.suggest_categorical("bidirectional", [False, True])
        dropout = trial.suggest_float("recurrentDropout", 0.0, 0.5)
        activationFunction = trial.suggest_categorical("encoderActivation", ["ReLU", "LeakyReLU", "GELU"])
        
        # Get input size from dataset info
        inputSize = datasetInfo["timeSeriesShape"][0]
        
        timeSeriesEncoderParams = {
            "activation_function": activationFunction,
            "layer_type": [
                {
                    "name": rnnType,
                    "params": {
                        "input_size": inputSize,
                        "hidden_size": hiddenDim,
                        "num_layers": numLayers,
                        "bias": True,
                        "batch_first": True,
                        "dropout": dropout if numLayers > 1 else 0.0,
                        "bidirectional": bidirectional,
                    }
                },
            ]
        }
        
    elif architectureType == "Conv1d":
        # Conv1d architecture
        numConvLayers = trial.suggest_int("numConvLayers", 1, 3)
        kernelSize = trial.suggest_categorical("kernelSize", [3, 5, 7])
        stride = trial.suggest_categorical("stride", [1, 2])
        activationFunction = trial.suggest_categorical("encoderActivation", ["ReLU", "LeakyReLU", "GELU"])
        
        # Start with input channels
        inputChannels = datasetInfo["timeSeriesShape"][0]
        
        # Build conv layers
        layerList = []
        currentChannels = inputChannels
        
        for i in range(numConvLayers):
            # Increase channels as we go deeper
            outChannels = trial.suggest_int(f"conv{i+1}_channels", 16, 128)
            
            layerList.append({
                "name": "Conv1d",
                "params": {
                    "in_channels": currentChannels,
                    "out_channels": outChannels,
                    "kernel_size": kernelSize,
                    "stride": stride,
                    "padding": kernelSize // 2,  # Same padding
                    "bias": True,
                }
            })
            
            # Add pooling after conv (except last layer)
            if i < numConvLayers - 1:
                poolType = trial.suggest_categorical(f"pool{i+1}_type", ["MaxPool1d", "AvgPool1d"])
                layerList.append({
                    "name": poolType,
                    "params": {
                        "kernel_size": 2,
                        "stride": 2,
                    }
                })
            
            currentChannels = outChannels
        
        # Add global pooling to reduce to fixed size
        layerList.append({
            "name": "AdaptiveAvgPool1d",
            "params": {
                "output_size": 1,
            }
        })
        
        # Flatten
        layerList.append({
            "name": "Flatten",
            "params": {}
        })
        
        timeSeriesEncoderParams = {
            "activation_function": activationFunction,
            "layer_type": layerList
        }
    
    architectureParameters["EncoderParams"] = timeSeriesEncoderParams
    return architectureParameters

In [6]:
def setUpFeedForwardHead(trial:optuna.Trial, architectureParameters:dict, datasetInfo:dict):
    """Setup the feedforward classification head"""
    # Calculate input size based on encoder outputs
    globalEmbeddingDim = architectureParameters["GlobalFFEncoderParams"]["layer_type"][0]["params"]["out_features"]
    
    # Determine encoder output size based on architecture type
    encoderParams = architectureParameters["EncoderParams"]
    
    # Check if this is RNN or Conv1d
    firstLayer = encoderParams["layer_type"][0]
    
    if firstLayer["name"] in ["LSTM", "GRU", "RNN"]:
        # RNN architecture
        hiddenDim = firstLayer["params"]["hidden_size"]
        bidirectional = firstLayer["params"]["bidirectional"]
        rnnOutputSize = hiddenDim * (2 if bidirectional else 1)
        encoderOutputSize = rnnOutputSize
        
    elif firstLayer["name"] == "Conv1d":
        # Conv1d architecture - find the last conv layer before pooling
        lastConvLayer = None
        for layer in encoderParams["layer_type"]:
            if layer["name"] == "Conv1d":
                lastConvLayer = layer
        
        if lastConvLayer:
            # After AdaptiveAvgPool1d(1) and Flatten, output size = out_channels
            encoderOutputSize = lastConvLayer["params"]["out_channels"]
        else:
            encoderOutputSize = 64  # Fallback
    else:
        # Fallback
        encoderOutputSize = 64
    
    combinedInputSize = encoderOutputSize + globalEmbeddingDim
    
    # Suggest feedforward head architecture
    numHiddenLayers = trial.suggest_int("numFFLayers", 1, 3)
    ffHiddenDim = trial.suggest_int("ffHiddenDim", 32, 256)
    ffDropout = trial.suggest_float("ffDropout", 0.0, 0.5)
    ffActivation = trial.suggest_categorical("ffActivation", ["ReLU", "LeakyReLU", "GELU"])
    
    # Build layer list - make sure last element is always Linear
    layerList_clean = []
    currentDim = combinedInputSize
    for i in range(numHiddenLayers):
        layerList_clean.append({
            "name": "Linear",
            "params": {
                "in_features": currentDim,
                "out_features": ffHiddenDim,
                "bias": True,
            }
        })
        # Add dropout BEFORE the next layer (not after the last one)
        if ffDropout > 0 and i < numHiddenLayers - 1:
            layerList_clean.append({
                "name": "Dropout",
                "params": {
                    "p": ffDropout,
                    "inplace": False,
                }
            })
        currentDim = ffHiddenDim
    
    # Note: Final output layer will be added by Direct/Autoencoder class
    # The last layer MUST have "out_features" for Direct to append the output layer
    feedForwardParams = {
        "activation_function": ffActivation,
        "layer_type": layerList_clean
    }
    
    architectureParameters["FeedForwardParams"] = feedForwardParams
    return architectureParameters

In [7]:
def setUpDecoder(trial:optuna.Trial, architectureParameters:dict, datasetInfo:dict):
    """Setup the decoder for autoencoder architecture (mirrors the encoder)"""
    encoderParams = architectureParameters["EncoderParams"]
    firstLayer = encoderParams["layer_type"][0]
    
    # Mirror the encoder architecture
    if firstLayer["name"] in ["LSTM", "GRU", "RNN"]:
        # RNN decoder
        rnnType = firstLayer["name"]
        hiddenDim = firstLayer["params"]["hidden_size"]
        numLayers = firstLayer["params"]["num_layers"]
        bidirectional = firstLayer["params"]["bidirectional"]
        dropout = firstLayer["params"]["dropout"]
        activationFunction = encoderParams["activation_function"]
        
        # Output should reconstruct the input
        outputSize = datasetInfo["timeSeriesShape"][0]
        # Note: seq_len is NOT a parameter for RNN layers - it's handled by the input data shape
        
        decoderParams = {
            "activation_function": activationFunction,
            "layer_type": [
                {
                    "name": rnnType,
                    "params": {
                        "input_size": hiddenDim * (2 if bidirectional else 1),
                        "hidden_size": outputSize,
                        "num_layers": numLayers,
                        "bias": True,
                        "batch_first": True,
                        "dropout": dropout if numLayers > 1 else 0.0,
                        "bidirectional": False,  # Decoder typically not bidirectional
                    }
                },
            ]
        }
        
    elif firstLayer["name"] == "Conv1d":
        # Conv1d decoder - symmetric architecture using ConvTranspose1d
        # The encoder ends with: Conv layers -> AdaptiveAvgPool1d(1) -> Flatten
        # Decoder: Unflatten -> ConvTranspose layers (reversed)
        
        # Get conv layers from encoder
        convLayers = [l for l in encoderParams["layer_type"] if l["name"] == "Conv1d"]
        poolLayers = [l for l in encoderParams["layer_type"] if l["name"] in ["MaxPool1d", "AvgPool1d"]]
        
        lastConvChannels = convLayers[-1]["params"]["out_channels"]
        kernelSize = convLayers[0]["params"]["kernel_size"]
        stride = convLayers[0]["params"]["stride"]
        
        # Calculate starting sequence length after encoder
        # After AdaptiveAvgPool1d(1), we have seq_len=1
        startSeqLen = 1
        
        # Calculate how many times we need to upsample
        numPoolLayers = len(poolLayers)
        
        layerList = []
        
        # First, unflatten from (batch, channels) to (batch, channels, 1)
        layerList.append({
            "name": "Unflatten",
            "params": {
                "dim": 1,
                "unflattened_size": (lastConvChannels, startSeqLen)
            }
        })
        
        # Reverse the conv layers
        reversedConvLayers = list(reversed(convLayers))
        
        # Build decoder layers symmetrically
        currentChannels = lastConvChannels
        for i, convLayer in enumerate(reversedConvLayers):
            # For the last layer, output should be original input channels (34)
            if i == len(reversedConvLayers) - 1:
                outChannels = datasetInfo["timeSeriesShape"][0]  # 34
            else:
                # Output channels should be input channels of the corresponding encoder layer
                outChannels = reversedConvLayers[i+1]["params"]["out_channels"]
            
            # Add upsampling with ConvTranspose1d
            # Use stride=2 to upsample if there was pooling in encoder
            useStride = 2 if i < numPoolLayers else stride
            
            layerList.append({
                "name": "ConvTranspose1d",
                "params": {
                    "in_channels": currentChannels,
                    "out_channels": outChannels,
                    "kernel_size": kernelSize,
                    "stride": useStride,
                    "padding": kernelSize // 2,
                    "output_padding": useStride - 1 if useStride > 1 else 0,
                    "bias": True,
                }
            })
            
            # Update current channels for next layer
            currentChannels = outChannels
        
        # Add final adjustment layer to match exact sequence length
        # Use adaptive interpolation if needed
        targetSeqLen = datasetInfo["timeSeriesShape"][1]  # 160
        layerList.append({
            "name": "AdaptiveAvgPool1d",
            "params": {
                "output_size": targetSeqLen,
            }
        })
        
        decoderParams = {
            "activation_function": encoderParams["activation_function"],
            "layer_type": layerList
        }
    else:
        # Fallback
        decoderParams = encoderParams
    
    # Mirror global decoder
    globalEmbeddingDim = architectureParameters["GlobalFFEncoderParams"]["layer_type"][0]["params"]["out_features"]
    globalDecoderParams = {
        "activation_function": "LeakyReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": globalEmbeddingDim,
                    "out_features": datasetInfo["globalFeaturesShape"][0],
                    "bias": True,
                }
            },
        ]
    }
    
    architectureParameters["DecoderParams"] = decoderParams
    architectureParameters["GlobalFFDecoderParams"] = globalDecoderParams
    return architectureParameters

## Weight Initialization

Apply He (Kaiming) initialization to improve training stability and convergence speed.

**Why He Initialization?**
- Designed specifically for ReLU-like activations
- Prevents vanishing/exploding gradients
- Helps the model converge faster
- Better than default PyTorch initialization for deep networks

**What's Applied:**
- **Linear & Conv1d layers**: Kaiming normal initialization (fan_in mode)
- **RNN layers**: Kaiming for input-hidden weights, orthogonal for hidden-hidden weights
- **Biases**: Small constant (0.01), with LSTM forget gate bias set to 1.0
- **Activation-aware**: Uses appropriate parameters for ReLU, LeakyReLU, and GELU

In [8]:
def apply_he_initialization(model, activation_type="ReLU"):
    """
    Apply He (Kaiming) initialization to all Linear and Conv1d layers in the model.
    
    He initialization is optimal for ReLU-like activations (ReLU, LeakyReLU).
    For GELU, it still works well as a general initialization strategy.
    
    Args:
        model: PyTorch model to initialize
        activation_type: Type of activation function ("ReLU", "LeakyReLU", "GELU")
    """
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            # He initialization for Linear layers
            if activation_type == "LeakyReLU":
                # For LeakyReLU, specify the negative slope (default 0.01)
                torch.nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='leaky_relu', a=0.01)
            elif activation_type in ["ReLU", "GELU"]:
                # For ReLU and GELU
                torch.nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
            
            # Initialize bias to small constant
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0.01)
                
        elif isinstance(module, torch.nn.Conv1d):
            # He initialization for Conv1d layers
            if activation_type == "LeakyReLU":
                torch.nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='leaky_relu', a=0.01)
            elif activation_type in ["ReLU", "GELU"]:
                torch.nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
            
            # Initialize bias to small constant
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0.01)
        
        elif isinstance(module, (torch.nn.LSTM, torch.nn.GRU)):
            # For RNN layers, initialize with orthogonal initialization (common practice)
            for param_name, param in module.named_parameters():
                if 'weight_ih' in param_name:
                    # Input-hidden weights: use He initialization
                    torch.nn.init.kaiming_normal_(param.data, mode='fan_in', nonlinearity='relu')
                elif 'weight_hh' in param_name:
                    # Hidden-hidden weights: use orthogonal initialization for stability
                    torch.nn.init.orthogonal_(param.data)
                elif 'bias' in param_name:
                    # Initialize biases to small constant
                    torch.nn.init.constant_(param.data, 0.01)
                    # For LSTM, set forget gate bias to 1 (helps with gradient flow)
                    if isinstance(module, torch.nn.LSTM):
                        n = param.data.size(0)
                        param.data[n//4:n//2].fill_(1.0)  # Forget gate bias
    
    print(f"✓ Applied He initialization (activation: {activation_type})")


In [9]:
def objective(trial: optuna.trial.Trial) -> float:
    """
    Objective function for Optuna optimization.
    Returns validation F1 score to maximize.
    """
    
    # Suggest macro architecture
    macroArchitecture = trial.suggest_categorical("MacroArchitecture", ["Direct", "Autoencoder"])
    
    # Setup data (include test for autoencoder, exclude for direct)
    includeTestInTrain = macroArchitecture == "Autoencoder"
    dataLoader.setup(stage='fit', includeTestInTrain=includeTestInTrain)
    trainLoader = dataLoader.train_dataloader()
    valLoader = dataLoader.val_dataloader()
    datasetInfo = dataLoader.getDatasetInfo()
    
    # Build architecture parameters
    archParams = {}
    
    # Setup encoders
    archParams = setUpEncoder(trial, archParams, datasetInfo)
    
    # Setup feedforward head
    archParams = setUpFeedForwardHead(trial, archParams, datasetInfo)
    
    # If autoencoder, setup decoder
    if macroArchitecture == "Autoencoder":
        archParams = setUpDecoder(trial, archParams, datasetInfo)
        # Add reconstruction loss weight
        archParams["ReconstructionLossWeight"] = trial.suggest_float("ReconstructionLossWeight", 0.1, 0.9)
    
    # Add common parameters
    archParams["OutputDim"] = 3  # no_pain, low_pain, high_pain
    archParams["LearningRate"] = trial.suggest_float("LearningRate", 1e-5, 1e-2, log=True)
    archParams["RegularizationWeight"] = trial.suggest_float("RegularizationWeight", 1e-6, 1e-2, log=True)
    archParams["Patience"] = trial.suggest_int("Patience", 3, 15)
    archParams["ClassWeightsPath"] = "../dataset/PirateProcessed/class_weights.yaml"
    
    # Create model based on architecture type
    if macroArchitecture == "Direct":
        model = Direct(archParams)
    else:  # Autoencoder
        model = LightningAutoencoder(archParams)
    
    # Apply He (Kaiming) initialization
    # Use the same activation as the feedforward head
    ff_activation = archParams["FeedForwardParams"]["activation_function"]
    apply_he_initialization(model, activation_type=ff_activation)
    
    # Training parameters
    max_epochs = trial.suggest_int("max_epochs", 20, 100)
    
    # Add early stopping callback
    early_stopping_callback = EarlyStopping(
        monitor='val_F1',
        patience=archParams["Patience"],
        mode='max',  # We want to maximize F1 score
        verbose=False
    )
    
    # Save best model checkpoint
    checkpoint_callback = ModelCheckpoint(
        monitor='val_F1',
        mode='max',
        save_top_k=1,
        filename=f'trial-{trial.number}-' + '{epoch:02d}-{val_F1:.3f}',
        verbose=False
    )
    
    # Create trainer
    trainer = Trainer(
        max_epochs=max_epochs,
        enable_progress_bar=False,
        enable_model_summary=False,
        log_every_n_steps=20,
        callbacks=[early_stopping_callback, checkpoint_callback],
        enable_checkpointing=True,
    )
    
    # Train the model
    try:
        trainer.fit(model, trainLoader, valLoader)
        
        # Get best validation F1 from checkpoint callback
        best_f1 = checkpoint_callback.best_model_score.item() if checkpoint_callback.best_model_score is not None else 0.0
        
        # Report for pruning
        trial.report(best_f1, step=trainer.current_epoch)
        
        # Handle pruning
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        return best_f1
        
    except Exception as e:
        raise optuna.TrialPruned() from e

## Run Optuna Optimization

Configure and run the hyperparameter search. We'll start with a modest number of trials to validate the setup.

**Benefits of Database Storage:**
- 💾 **Persistence**: Results survive notebook restarts
- 🔄 **Resume**: Continue optimization from where you left off
- 🚀 **Parallel**: Run multiple optimization processes simultaneously
- 📊 **Analysis**: Access results from any notebook or script

In [10]:
# Create Optuna study with database storage
study = optuna.create_study(
    direction='maximize',  # Maximize F1 score
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=5,
        interval_steps=1
    ),
    study_name='MATTEO_pirate_pain_multi_architecture',
    storage=storage,
    load_if_exists=True  # Resume from existing study if available
)

print("✓ Study created/loaded successfully!")
print(f"Study name: {study.study_name}")
print(f"Sampler: {study.sampler.__class__.__name__}")
print(f"Pruner: {study.pruner.__class__.__name__}")
print(f"Storage: {'Database' if storage else 'In-memory'}")
print(f"Total trials: {len(study.trials)}")
if len(study.trials) > 0:
    try:
        print(f"Best trial so far: {study.best_trial.number}")
        print(f"Best F1 score: {study.best_value:.4f}")
    except Exception:
        print("No best yet")

[I 2025-11-12 21:13:53,634] A new study created in RDB with name: MATTEO_pirate_pain_multi_architecture


✓ Study created/loaded successfully!
Study name: MATTEO_pirate_pain_multi_architecture
Sampler: TPESampler
Pruner: MedianPruner
Storage: Database
Total trials: 0


## Optional: Load Existing Study from Database

If you want to analyze results from a previous run without creating a new study, use this cell instead of the one below.

In [10]:
# Load existing study from database (alternative to creating new one)
# Uncomment and run this instead of the cell below if you want to just analyze existing results

# study = optuna.load_study(
#     study_name='pirate_pain_multi_architecture',
#     storage=storage
# )
# 
# print(f"✓ Study loaded from database!")
# print(f"Study name: {study.study_name}")
# print(f"Total trials: {len(study.trials)}")
# if len(study.trials) > 0:
#     print(f"Best F1 score: {study.best_value:.4f}")

In [11]:
# Run optimization
# Start with a small number of trials to validate setup
 # Increase this for longer runs

print(f"Starting optimization ...")
print("This may take a while depending on your hardware.")
print("-" * 60)

study.optimize(objective, show_progress_bar=True)

print("\nOptimization completed!")
print(f"Best trial: {study.best_trial.number}")
print(f"Best F1 score: {study.best_value:.4f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Starting optimization ...
This may take a while depending on your hardware.
------------------------------------------------------------
✓ Applied He initialization (activation: ReLU)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
[I 2025-11-12 21:27:42,389] Trial 0 pruned. 


✓ Applied He initialization (activation: LeakyReLU)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
[I 2025-11-12 21:28:04,659] Trial 1 finished with value: 0.8560606241226196 and parameters: {'MacroArchitecture': 'Direct', 'architectureType': 'Conv1d', 'numConvLayers': 1, 'kernelSize': 3, 'stride': 2, 'encoderActivation': 'ReLU', 'conv1_channels': 93, 'numFFLayers': 2, 'ffHiddenDim': 59, 'ffDropout': 0.2475884550556351, 'ffActivation': 'LeakyReLU', 'LearningRate': 0.0009717775305059633, 'RegularizationWeight': 1.7654048052495086e-05, 'Patience': 9, 'max_epochs': 64}. Best is trial 1 with value: 0.85606062412262.


✓ Applied He initialization (activation: LeakyReLU)


...

✓ Applied He initialization (activation: GELU)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
[I 2025-11-12 22:32:55,163] Trial 50 finished with value: 0.8787878751754761 and parameters: {'MacroArchitecture': 'Direct', 'architectureType': 'Recurrent', 'rnnType': 'GRU', 'hiddenDim': 198, 'numLayers': 2, 'bidirectional': False, 'recurrentDropout': 0.38747083995618553, 'encoderActivation': 'GELU', 'numFFLayers': 2, 'ffHiddenDim': 233, 'ffDropout': 0.3379274738393601, 'ffActivation': 'GELU', 'LearningRate': 0.0016331473610061117, 'RegularizationWeight': 0.0009200307325853929, 'Patience': 7, 'max_epochs': 47}. Best is trial 18 with value: 0.94696968793869.


✓ Applied He initialization (activation: ReLU)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
[I 2025-11-12 22:34:10,460] Trial 51 pruned. 


KeyboardInterrupt: 

## Visualize Results

Now let's analyze the optimization results to understand which hyperparameters had the most impact.

In [12]:
# Optimization history
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate

# Plot optimization history
fig = plot_optimization_history(study)
fig.show()

# Plot parameter importances
fig = plot_param_importances(study)
fig.show()

# Plot parallel coordinate (shows relationship between hyperparameters and objective value)
fig = plot_parallel_coordinate(study)
fig.show()

ImportError: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.

## Study Status and Database Info

Check the current status of trials stored in the database.

In [13]:
# Check study status from database
print(f"Study name: {study.study_name}")
print(f"Direction: {study.direction}")
print(f"Total trials: {len(study.trials)}")
print(f"Completed trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}")
print(f"Failed trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])}")
print(f"Pruned trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}")
print(f"Running trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.RUNNING])}")

if len(study.trials) > 0:
    completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if completed_trials:
        print(f"\n✓ Best trial: {study.best_trial.number}")
        print(f"✓ Best F1 score: {study.best_value:.4f}")
        print(f"\nTop 5 trials:")
        sorted_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:5]
        for i, trial in enumerate(sorted_trials, 1):
            arch = trial.params.get('MacroArchitecture', 'Unknown')
            enc = trial.params.get('architectureType', 'Unknown')
            print(f"  {i}. Trial {trial.number}: F1={trial.value:.4f} | {arch} | {enc}")
    
    print("\n📊 Trial states (last 10):")
    for trial in study.trials[-10:]:
        state_symbol = "✓" if trial.state == optuna.trial.TrialState.COMPLETE else "✗" if trial.state == optuna.trial.TrialState.FAIL else "⊗" if trial.state == optuna.trial.TrialState.PRUNED else "⟳"
        value_str = f"F1={trial.value:.4f}" if trial.value is not None else "N/A"
        print(f"  {state_symbol} Trial {trial.number}: {trial.state.name} | {value_str}")
else:
    print("\n⚠️ No trials found in this study. Run optimization to start!")

Study name: MATTEO_pirate_pain_multi_architecture
Direction: 2
Total trials: 52
Completed trials: 44
Failed trials: 0
Pruned trials: 8
Running trials: 0

✓ Best trial: 18
✓ Best F1 score: 0.9470

Top 5 trials:
  1. Trial 18: F1=0.9470 | Direct | Conv1d
  2. Trial 24: F1=0.9470 | Direct | Conv1d
  3. Trial 19: F1=0.9394 | Direct | Conv1d
  4. Trial 34: F1=0.9394 | Direct | Conv1d
  5. Trial 40: F1=0.9394 | Autoencoder | Conv1d

📊 Trial states (last 10):
  ✓ Trial 42: COMPLETE | F1=0.8864
  ✓ Trial 43: COMPLETE | F1=0.9394
  ⊗ Trial 44: PRUNED | F1=0.9015
  ✓ Trial 45: COMPLETE | F1=0.8636
  ⊗ Trial 46: PRUNED | N/A
  ✓ Trial 47: COMPLETE | F1=0.7045
  ✓ Trial 48: COMPLETE | F1=0.9091
  ✓ Trial 49: COMPLETE | F1=0.9318
  ✓ Trial 50: COMPLETE | F1=0.8788
  ⊗ Trial 51: PRUNED | F1=0.9242


## Load and Evaluate Best Model

Load the best checkpoint and evaluate it on the validation set.

In [22]:
# Reconstruct the best model from best trial parameters
from pytorch_lightning import Trainer
from torchmetrics import ConfusionMatrix
from GradientGang.Pipeline.Architectures.LightningAutoencoder import LightningAutoencoder
from GradientGang.Pipeline.Architectures.Direct import Direct
import torch

best_params = study.best_params
best_trial_from_optuna = study.best_trial.number

print("Reconstructing best model with parameters:")
print(f"Best trial from Optuna: {best_trial_from_optuna}")
for key, value in best_params.items():
    print(f"  {key}: {value}")

# Create a dummy trial to reuse setup functions
class BestTrial:
    def __init__(self, params):
        self.params = params
    
    def suggest_categorical(self, name, choices):
        return self.params[name]
    
    def suggest_int(self, name, low, high, log=False):
        return self.params[name]
    
    def suggest_float(self, name, low, high, log=False):
        return self.params[name]

best_trial = BestTrial(best_params)

# Reconstruct architecture
archParams = {}
archParams = setUpEncoder(best_trial, archParams, dataLoader.getDatasetInfo())
archParams = setUpFeedForwardHead(best_trial, archParams, dataLoader.getDatasetInfo())

# Check if autoencoder
macroArchitecture = best_params.get('MacroArchitecture', 'Direct')
if macroArchitecture == "Autoencoder":
    archParams = setUpDecoder(best_trial, archParams, dataLoader.getDatasetInfo())
    archParams["ReconstructionLossWeight"] = best_params['ReconstructionLossWeight']

archParams['LearningRate'] = best_params['LearningRate']
archParams['RegularizationWeight'] = best_params['RegularizationWeight']
archParams['Patience'] = best_params['Patience']
archParams['OutputDim'] = dataLoader.getDatasetInfo()['numClasses']
archParams['ClassWeightsPath'] = '../dataset/PirateProcessed/class_weights.yaml'

# Find the checkpoint for the best trial (search ALL version directories)
import os
import glob
import re

print(f"\nSearching for checkpoint from trial {best_trial_from_optuna}...")

# Get ALL version directories
log_dirs = glob.glob('lightning_logs/version_*')
print(f"Found {len(log_dirs)} log directories")

best_checkpoint_path = None
best_checkpoint_f1 = 0.0

# Search through ALL checkpoints in ALL directories
for log_dir in log_dirs:
    checkpoints = glob.glob(os.path.join(log_dir, 'checkpoints', '*.ckpt'))
    for checkpoint in checkpoints:
        # Extract trial number from checkpoint filename
        # Format: trial-{number}-epoch={epoch}-val_F1={f1}.ckpt
        trial_match = re.search(r'trial-(\d+)', checkpoint)
        f1_match = re.search(r'val_F1=([\d.]+)\.ckpt', checkpoint)
        
        if trial_match:
            trial_num = int(trial_match.group(1))
            
            # Check if this checkpoint is from the best trial
            if trial_num == best_trial_from_optuna:
                if f1_match:
                    f1_score = float(f1_match.group(1))
                    # Keep the checkpoint with highest F1 for this trial
                    if f1_score > best_checkpoint_f1:
                        best_checkpoint_path = checkpoint
                        best_checkpoint_f1 = f1_score

if best_checkpoint_path:
    print(f"\n✓ Found checkpoint for trial {best_trial_from_optuna}")
    print(f"Checkpoint path: {best_checkpoint_path}")
    print(f"Checkpoint F1 score: {best_checkpoint_f1:.4f}")
    
    # Load appropriate model type
    if macroArchitecture == "Direct":
        model = Direct.load_from_checkpoint(best_checkpoint_path, params=archParams)
    else:
        model = LightningAutoencoder.load_from_checkpoint(best_checkpoint_path, params=archParams)
    
    # Evaluate on validation set
    trainer = Trainer(logger=False, enable_checkpointing=False)
    val_results = trainer.validate(model, datamodule=dataLoader)
    
    print(f"\nValidation Results:")
    print(f"  F1 Score: {val_results[0]['val_F1']:.4f}")
    print(f"  Loss: {val_results[0]['val_loss']:.4f}")
else:
    print(f"\n✗ No checkpoint found for trial {best_trial_from_optuna}!")
    print(f"This trial's checkpoint may have been deleted or not saved.")
    print(f"You can re-run trial {best_trial_from_optuna} with these parameters to recreate it.")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Reconstructing best model with parameters:
Best trial from Optuna: 18
  MacroArchitecture: Direct
  architectureType: Conv1d
  numConvLayers: 2
  kernelSize: 3
  stride: 1
  encoderActivation: GELU
  conv1_channels: 90
  pool1_type: AvgPool1d
  conv2_channels: 97
  numFFLayers: 2
  ffHiddenDim: 184
  ffDropout: 0.219781104328841
  ffActivation: GELU
  LearningRate: 0.000503935772382754
  RegularizationWeight: 0.000967587574486895
  Patience: 12
  max_epochs: 46

Searching for checkpoint from trial 18...
Found 52 log directories

✓ Found checkpoint for trial 18
Checkpoint path: lightning_logs\version_18\checkpoints\trial-18-epoch=24-val_F1=0.947.ckpt
Checkpoint F1 score: 0.9470


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Reconstructing best model with parameters:
Best trial from Optuna: 18
  MacroArchitecture: Direct
  architectureType: Conv1d
  numConvLayers: 2
  kernelSize: 3
  stride: 1
  encoderActivation: GELU
  conv1_channels: 90
  pool1_type: AvgPool1d
  conv2_channels: 97
  numFFLayers: 2
  ffHiddenDim: 184
  ffDropout: 0.219781104328841
  ffActivation: GELU
  LearningRate: 0.000503935772382754
  RegularizationWeight: 0.000967587574486895
  Patience: 12
  max_epochs: 46

Searching for checkpoint from trial 18...
Found 52 log directories

✓ Found checkpoint for trial 18
Checkpoint path: lightning_logs\version_18\checkpoints\trial-18-epoch=24-val_F1=0.947.ckpt
Checkpoint F1 score: 0.9470


...

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Reconstructing best model with parameters:
Best trial from Optuna: 18
  MacroArchitecture: Direct
  architectureType: Conv1d
  numConvLayers: 2
  kernelSize: 3
  stride: 1
  encoderActivation: GELU
  conv1_channels: 90
  pool1_type: AvgPool1d
  conv2_channels: 97
  numFFLayers: 2
  ffHiddenDim: 184
  ffDropout: 0.219781104328841
  ffActivation: GELU
  LearningRate: 0.000503935772382754
  RegularizationWeight: 0.000967587574486895
  Patience: 12
  max_epochs: 46

Searching for checkpoint from trial 18...
Found 52 log directories

✓ Found checkpoint for trial 18
Checkpoint path: lightning_logs\version_18\checkpoints\trial-18-epoch=24-val_F1=0.947.ckpt
Checkpoint F1 score: 0.9470


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val_F1           │    0.9469696879386902     │
│         val_loss          │    0.5885179042816162     │
│    val_prediction_loss    │    0.5885179042816162     │
└───────────────────────────┴───────────────────────────┘


Validation Results:
  F1 Score: 0.9470
  Loss: 0.5885


## Optional: Generate Submission

If you want to generate predictions for the test set, run this cell.

In [24]:
# Generate submission using SubmissionGenerator
if best_checkpoint_path and 'model' in locals():
    from GradientGang.Pipeline.SubmissionGenerator.SubmissionGenerator import SubmissionGenerator
    from datetime import datetime
    
    # Setup test data
    dataLoader.setup(stage='test')
    testLoader = dataLoader.test_dataloader()
    
    # Create submission path with trial number
    timestamp = datetime.now().strftime("%H-%M")
    path = f"../Submissions/submission_MATTEO_{timestamp}.csv"
    
    # Create submission generator
    submission_generator = SubmissionGenerator(
        model=model,
        dataloader=testLoader,
        label_mapping={0: 'no_pain', 1: 'low_pain', 2: 'high_pain'},
    )
    
    # Generate submission
    submission_generator.generate_submission(output_path=path)
    
    print(f"\n✓ Submission saved to: {path}")
else:
    print("⚠️ No model loaded. Please run the previous cell to load the best checkpoint first.")


✓ Submission saved to: ../Submissions/submission_MATTEO_22-47.csv
